# Heritage Short: music stage (ACE-Step 1.5 on Kaggle), v2

Kaggle's kernel is Python 3.13, but ACE-Step 1.5 requires Python 3.11–3.12 (FAILURES.md F11).
So this notebook does **not** install ACE-Step into the kernel: it uses ACE-Step's own documented
setup (`uv sync`, which provisions Python 3.12 in the repo's `.venv`) and runs generation as a script there.

**Settings:** Accelerator → **GPU T4 x2**, Internet → **On**. Then **Run All**.
Expect ~5–10 min of setup, then well under a minute per 90-second track.

In [ ]:
# 1. The storyboard's music block (verbatim copy of storyboard.json -> music)
import json, time, os
MUSIC = json.loads(r'''{
  "model": "ACE-Step 1.5",
  "caption": "nostalgic intimate instrumental film score, Ethiopian-inspired pentatonic melody in the feel of tizita, solo krar plucks and a soft masinko line, warm felt piano, gentle string pad, sparse and slow, no drums, warm analog texture, peaceful and wistful",
  "instrumental": true,
  "bpm": 70,
  "key": "D Minor",
  "duration_s": 90,
  "candidates": 3,
  "seeds": [
    3001,
    3002,
    3003
  ]
}''')
os.makedirs("/kaggle/working/music", exist_ok=True)
json.dump(MUSIC, open("/kaggle/working/music_cfg.json", "w"), ensure_ascii=False)
json.dump({"t_start": time.time()}, open("/kaggle/working/session_t0.json", "w"))
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

In [ ]:
# 2. Install ACE-Step 1.5 with its own Python 3.12 (uv), as its README documents
t = time.time()
!pip install -q uv
!git clone -q https://github.com/ACE-Step/ACE-Step-1.5.git /kaggle/working/ACE-Step-1.5 2>/dev/null || echo "already cloned"
!cd /kaggle/working/ACE-Step-1.5 && uv sync 2>&1 | tail -4
!cd /kaggle/working/ACE-Step-1.5 && uv run python -c "import sys, torch; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"
json.dump({"install_s": round(time.time() - t)}, open("/kaggle/working/session_install.json", "w"))
print(f"install: {time.time() - t:.0f}s")

In [ ]:
%%writefile /kaggle/working/gen.py
# Runs INSIDE ACE-Step's Python 3.12 env: uv run python /kaggle/working/gen.py
import inspect, json, subprocess, sys, time
import torch
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

PROJECT = "/kaggle/working/ACE-Step-1.5"
OUT = "/kaggle/working/music"
MUSIC = json.load(open("/kaggle/working/music_cfg.json"))

t = time.time()
try:
    dit = AceStepHandler()
    dit.initialize_service(project_root=PROJECT, config_path="acestep-v15-turbo", device="cuda")
except Exception as e:
    print("DiT init failed:", repr(e))
    print("initialize_service signature:", inspect.signature(AceStepHandler.initialize_service))
    sys.exit(1)
llm = LLMHandler()
try:  # the LM planner is optional for an instrumental with exact metadata
    llm.initialize(checkpoint_dir=f"{PROJECT}/checkpoints", lm_model_path="acestep-5Hz-lm-0.6B",
                   backend="pt", device="cuda")
except Exception as e:
    print("LM planner not loaded (continuing with thinking=False):", repr(e)[:200])
models_s = time.time() - t
print(f"models loaded in {models_s:.0f}s")

def make_params(seed):
    kw = dict(caption=MUSIC["caption"], instrumental=True, duration=MUSIC["duration_s"],
              bpm=MUSIC["bpm"], keyscale=MUSIC["key"], seed=seed)
    try:
        return GenerationParams(**kw, thinking=False)
    except TypeError:
        return GenerationParams(**kw)

runs = []
for i, seed in enumerate(MUSIC["seeds"]):
    t = time.time()
    res = generate_music(dit, llm, make_params(seed),
                         GenerationConfig(batch_size=1, use_random_seed=False, audio_format="flac"),
                         save_dir=f"{OUT}/raw_{i}")
    lat = time.time() - t
    rec = {"ts": time.strftime("%Y-%m-%dT%H:%M:%S+00:00", time.gmtime()), "stage": "music",
           "model": "ACE-Step 1.5 (acestep-v15-turbo)", "shot": None,
           "params": {"caption": MUSIC["caption"], "bpm": MUSIC["bpm"], "key": MUSIC["key"],
                      "duration_s": MUSIC["duration_s"], "instrumental": True, "candidate": i},
           "seed": seed, "latency_s": round(lat, 2), "cache_hit": False, "cost_usd": 0.0,
           "gpu": torch.cuda.get_device_name(0), "gpu_minutes": round(lat / 60, 3)}
    if not res.success:
        rec.update(status="error", error=str(res.error)[:300])
        print(f"candidate {i} FAILED: {res.error}")
    else:
        dst = f"{OUT}/candidate_{i}.wav"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", res.audios[0]["path"],
                        "-ar", "48000", dst], check=True)
        rec.update(status="ok", output=f"outputs/music/candidate_{i}.wav")
        print(f"candidate {i}: seed {seed}, {lat:.1f}s -> {dst}")
    runs.append(rec)

with open(f"{OUT}/runs_music.jsonl", "w") as f:
    for r in runs:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
json.dump({"models_load_s": round(models_s), "gpu": torch.cuda.get_device_name(0),
           "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)},
          open(f"{OUT}/gen_session.json", "w"), indent=2)


In [ ]:
# 3. Generate the candidates (first run also downloads the model weights)
# MPLBACKEND=Agg: Kaggle exports an inline matplotlib backend that the isolated venv lacks (F12)
!cd /kaggle/working/ACE-Step-1.5 && MPLBACKEND=Agg uv run python /kaggle/working/gen.py

In [ ]:
# 4. Pack results for download
s = {}
for p in ["session_install.json", "music/gen_session.json"]:
    if os.path.exists(f"/kaggle/working/{p}"):
        s.update(json.load(open(f"/kaggle/working/{p}")))
s["total_session_s"] = round(time.time() - json.load(open("/kaggle/working/session_t0.json"))["t_start"])
json.dump(s, open("/kaggle/working/music/session.json", "w"), indent=2)
!cd /kaggle/working && rm -rf music/raw_* && zip -qr music_outputs.zip music && ls -la music
print(s)
print("Download /kaggle/working/music_outputs.zip from the Output panel on the right.")